# 40 · Relate, allowing for spelling, checked against pandas, rapidfuzz and jellyfish

## What this checks

**Relate a column from another table** matches rows by name. This check covers release 0.15.0, and its
option to allow for spelling.

Two tables often spell the same place differently. With this option, Relate first folds each name into a plain form
for comparing. It drops the words it is told to drop, as whole words. Then a name with no exact match is
matched to the closest spelling, if it scores 0.88 or more, and no other name is within 0.02 of it. The score is
Jaro-Winkler, a measure of how alike two spellings are, from 0 to 1.

A wrong match puts one district's numbers on another.

## How it is checked

We repeat every step with tools that share no code with Fieldwork:

- **Folding** is rewritten here from Fieldwork's stated rule, with Python's own Unicode tools.
- **Exact matches** are a join in **pandas**, Python's library for tables, on the folded names.
- **Spelling scores** come from **rapidfuzz**, written in C++, with **jellyfish**, written in Rust, beside it.
- **The related table** is a pandas join on the matches.

This notebook runs in CPython only: rapidfuzz and jellyfish are not in the browser kernel. In the browser, the
notebook says so and skips its later cells.

The data are Sierra Leone's 2019 measles cases by home district. The line list writes "Bo District". The population
table uses OCHA's district names, which write "Bo", and "Fabala" for Falaba. Three cases are run:

1. "District" dropped, spelling allowed down to 0.88, the default.
2. The same at 0.96. Falaba's spelling scores about 0.956, so it falls below.
3. 0.88 again, with nothing dropped.

## What should happen

1. **Every folded name** is exactly what Fieldwork shows as "compared as".
2. **Every outcome and match** is the same: exact, spelling, ambiguous or none, and the value matched. The outcome is
   decided from rapidfuzz's score, under Fieldwork's stated rule.
3. **Every spelling score** is within 10⁻⁹ of rapidfuzz's.
4. **Every row of the related table** is the same. The district brought across equals pandas' join, the unmatched
   rows sit under the case's label, and the number of rows does not change.

We also record, without judging, where jellyfish differs from rapidfuzz. And we record any score within 10⁻⁹ of the
threshold, where a computer's rounding could tip it either way.

These criteria were fixed before the check was run.

In [ ]:
# Only what this notebook imports, requested one at a time, so that a package the browser kernel lacks does not stop
# the others from loading. See 00-kernel-check for what this kernel really has.
NEEDED = ["pandas", "rapidfuzz", "jellyfish"]
try:
    import pyodide_js
except ImportError:
    pyodide_js = None
    print("Not a Pyodide kernel; using the ambient environment.")
if pyodide_js:
    for package in NEEDED:
        try:
            await pyodide_js.loadPackage(package)
            print("Loaded:", package)
        except Exception:
            print("Not in this kernel:", package)

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd()))
try:
    from checks.check_40_relate_spelling import run
except ImportError as missing:
    report = None
    print("Not runnable here:", missing)
else:
    report = run(Path.cwd() / "fixtures" / "relate-spelling-sl-001.json")
    print("agrees:", report["agrees"])
    for c in report["comparisons"]:
        print(f"\n{'OK ' if c['agrees'] else 'DIFF'} case {c['case']}: {c['note']}")
        print(f"  outcomes {c['outcomes']}; {c['rows']} rows, {'equal' if c['rowsAgree'] else 'NOT equal'} to the pandas merge")
        for n in c["names"]:
            if n["fieldwork"]["outcome"] != "exact" or not n["agrees"]:
                p, f = n["pandas"], n["fieldwork"]
                print(f"  {'OK ' if n['agrees'] else 'DIFF'} {n['name']}: {p['outcome']} {p['value'] or ''} {p['score'] or ''}"
                      f" / {f['outcome']} {f['value'] or ''} {f['score'] or ''}")

## The result on record

First recorded on 9 November 2026, against release 0.10.0. Rerun on 26 November 2026, against release 0.15.0:
**Agrees**.

All three cases agree on every one of the 17 rows:

- **Case 1.** 15 names match exactly. Falaba matches "Fabala" by spelling, with a score of about 0.956. The blank
  is kept under a named label.
- **Case 2.** At 0.96, the same 15 match exactly, and Falaba finds no match.
- **Case 3.** With "District" kept, 13 names still match by spelling, and 3 find none.

jellyfish gave the same scores as rapidfuzz everywhere.

Running the cell above repeats the check on the same data.

## On the threshold

A score equal to the threshold passes. In case 3, four names score exactly 0.88: Bonthe, Kambia, Karene and Kenema
Districts. There, a computer's rounding on either side could tip the result. The check names them, and both sides
passed them.

In [ ]:
if report is None:
    print("Skipped: this check is not runnable in this kernel (see above).")
else:
    print("on the threshold:", [f"case {t['case']}: {t['name']}" for t in report["onThreshold"]])
    print("jellyfish differs from rapidfuzz:", report["jellyfishDiffers"] or "nowhere")

## What this does not show

- That a spelling match is the same place. "Fabala" is Falaba because OCHA's list holds no other district like it.
  A person sees that in Relate's preview. The step cannot.
- That 0.88 suits district names. Case 3 shows that, with "District" kept, 13 of 16 names still pass, on the shared
  prefix alone. The words dropped do more of the matching than the threshold does.

In [ ]:
if report is None:
    print("Skipped: this check is not runnable in this kernel (see above).")
else:
    for item in report["notEstablished"]:
        print("-", item)
    print()
    print("fixture :", report["fixture"]["id"], report["fixture"]["sha256"][:16])
    print("pandas  :", report["environment"]["pandas"], "· rapidfuzz", report["environment"]["rapidfuzz"], "· jellyfish", report["environment"]["jellyfish"])
    print("runtime :", report["environment"]["runtime"])